# Project 1: Neural Decoding for a Brain Computer Interface

You just joined a computational neuroscience lab working on a brain computer interface. The team is developing a minimalist neural prosthetic: a system that reads brain activity and decodes the user's intent in real time.

A colleague has left you a dataset: spike recordings from a population of neurons, captured while a subject was exposed to two distinct visual stimuli. Your mission is to build the decoding component, the module that sits between the electrode array and the downstream system, to answer one question: **which stimulus is the brain currently responding to?**

You will implement the simplest viable decoder: a Perceptron, from scratch, in NumPy. You will make it work, then you will do what any serious lab member does before shipping a model: interrogate it!

The PI's question is not "does it work?". Of course it does. Her question is: **what does this model actually say about how neurons encode information, and where does it break down as a biological account?**

You will write your findings as a short article, and present them at the next lab meeting.

## Your course toolbox

Everything you need is in the course. When you are stuck, go back to the chapter:

| Step of this project | Where it is in the course |
|:---|:---|
| Read the raw CSV files | `data/README.md`, and `preview()` in `src/spikes.py` |
| Read rasters and PSTHs | Chapter 3: *Reading Spike Data: Raster and PSTH* |
| Spikes to counts to firing rates (`sum` over an axis) | Chapter 3: *Warm-Up*, `notebooks/v2/Chapter3_LiveExercises.ipynb` |
| Rate coding, and what it discards | Chapter 3: *Rate Is Not the Only Code* |
| The perceptron rule and the training loop | Chapter 5, `notebooks/v2/Chapter5_LiveExercises.ipynb` |
| Train / test split, accuracy, confusion matrix | Chapter 5: *Evaluating a Decoder* |
| Standardize features, silent neurons | Chapter 5: *Real Recordings: Standardize Your Features* |
| Read the weights as biology | Chapter 5: *Chapter Bilan: Decode a Stimulus From Firing Rates* |
| Kept vs discarded | The epistemological bridge, every chapter |

---

## 0. Setup

Run `uv sync` once at the repository root, then open this notebook with the project kernel. Everything it needs is in this folder. The recordings are CSV files in the `data/` folder next to this notebook; `src/spikes.py` loads them and provides two plotting helpers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.spikes import load_dataset, plot_raster, plot_psth

spikes, labels, label_names, dt = load_dataset("bci_visual_2stim")
n_trials, n_neurons, n_bins = spikes.shape

print(f"{n_trials} trials, {n_neurons} neurons, {n_bins} bins of {dt * 1000:.0f} ms")
print("classes:", label_names, "| trials per class:", np.bincount(labels))

### What is in the dataset?

The recordings are **CSV files** in `data/`, described in `data/README.md`. Open them: any spreadsheet or text editor will do.

| File | One row per | Columns |
|:---|:---|:---|
| `datasets.csv` | recording session | number of trials and neurons, trial duration, time resolution, stimulus onset |
| `<dataset>_trials.csv` | trial | `trial`, `label`, `label_name` |
| `<dataset>_spikes.csv` | spike | `trial`, `neuron`, `time_ms` |

`load_dataset(name)` turns them into arrays:

| Variable | Shape | Meaning |
|:---|:---|:---|
| `spikes` | `(n_trials, n_neurons, n_bins)` | `1` if the neuron fired in that 1 ms bin, `0` otherwise |
| `labels` | `(n_trials,)` | the class of each trial |
| `label_names` | `(n_classes,)` | human-readable names of the classes |
| `dt` | scalar | bin width in seconds |

Each trial lasts 1 s. The stimulus appears at **t = 200 ms**.

**Have a look at the raw files** before anything else: the cell below prints the first lines of each one.

In [ ]:
from src.spikes import preview

preview("bci_visual_2stim")

---

## 1. Look at the data before modelling it

A raster plot shows one tick per spike: one row per trial, coloured by stimulus. A PSTH (peri-stimulus time histogram) averages those trials into a firing rate over time.

In [ ]:
_neuron = 0 # Change here to study another neuron

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_raster(spikes, labels, neuron=_neuron, label_names=label_names, dt=dt, ax=axes[0])
plot_psth(spikes, labels, neuron=_neuron, label_names=label_names, dt=dt, ax=axes[1])
plt.tight_layout()

**Your turn.** Browse through the neurons (change `neuron=`). Before writing any model, write down in the cell below:

- Do all neurons respond to the stimuli? Do they all respond the same way?
- For a given neuron, is the response identical from one trial to the next?
- What information could a decoder use to tell the two stimuli apart?

*Your observations here.*

---

## 2. From spike trains to feature vectors

The perceptron takes a vector $x \in \mathbb{R}^n$. We need to turn each trial (a `n_neurons × n_bins` matrix) into one vector.

The simplest choice is **rate coding**: count the spikes of each neuron during the trial (Chapter 3).

**Your turn.**
1. Build `counts`, a `(n_trials, n_neurons)` matrix of spike counts.
2. Standardize each column (zero mean, unit variance), as in Chapter 5. Compute the mean and the standard deviation on the **training trials only** (the split is done for you below).
3. **Be careful:** Some neurons never fire: their standard deviation is 0. Decide what to do with them, and explain why.

In [ ]:
rng = np.random.default_rng(42) # DON'T EDIT THIS FOR REPRODUCIBILITY
order = rng.permutation(n_trials)
n_train = int(0.7 * n_trials)
train, test = order[:n_train], order[n_train:]   # 70 % to learn, 30 % to measure

# TODO: spike count of each neuron in each trial -> shape (n_trials, n_neurons)
counts = ...

# TODO: standardize each feature with the TRAINING mean and standard deviation
X = ...

print(X.shape)

---

## 3. A Perceptron from scratch

Recall the neuron and the rule from the course, for an input $x$, a target $t \in \{0, 1\}$ and a learning rate $\eta$:

$$z = w \cdot x + b \qquad y = \varphi(z) = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{otherwise} \end{cases} \qquad w \leftarrow w + \eta\,(t - y)\,x \qquad b \leftarrow b + \eta\,(t - y)$$

**Your turn.** Complete the class below. Only NumPy is allowed. Visiting the training trials in a **random order** at each epoch is a good habit (why?).

In [ ]:
class Perceptron:
    def __init__(self, n_features, eta=0.1, epochs=20, seed=0):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.eta = eta
        self.epochs = epochs
        self.rng = np.random.default_rng(seed)
        self.mistakes_ = []  # number of mistakes per epoch

    def predict(self, X):
        # TODO: return an array of 0/1 predictions, one per row of X
        raise NotImplementedError

    def fit(self, X, T):
        # TODO: loop over epochs, visit the samples in a random order,
        # apply the update rule, record the number of mistakes per epoch,
        # and stop after an epoch without any mistake
        raise NotImplementedError
        return self

---

## 4. Train and evaluate

Never evaluate a decoder on the trials it was trained on (Chapter 5).

In [ ]:
model = Perceptron(n_features=X.shape[1]).fit(X[train], labels[train])

train_acc = np.mean(model.predict(X[train]) == labels[train])
test_acc = np.mean(model.predict(X[test]) == labels[test])
print(f"train accuracy: {train_acc:.1%} | test accuracy: {test_acc:.1%}")

plt.plot(model.mistakes_, marker="o")
plt.xlabel("epoch")
plt.ylabel("mistakes")
plt.title("Perceptron learning curve");

**Your turn.** Compute the **confusion matrix** on the test trials (rows: true stimulus, columns: prediction).

In [ ]:
# TODO: confusion matrix on the test trials

---

## 5. Interrogate the model

It works. Now answer the PI's question: **what does this model say about how neurons encode information, and where does it break down as a biological account?**

Some leads to investigate (you do not have to follow all of them, and you are welcome to add your own):

1. **Read the weights.** Plot `model.w` per neuron. Compare it to how strongly each neuron's spike count differs between the two stimuli. Does a large weight mean a neuron "cares" about the stimulus? What about small weights? *(Remember: the perceptron stops as soon as the training set is separated.)*
2. **Neurons the model ignores.** Pick neurons with a near-zero count difference and look at their rasters and PSTHs. Is there really no information there?
3. **Robustness.** What happens to test accuracy if you keep only a subset of neurons, fewer training trials, or a shorter counting window?
4. **Biology.** List the assumptions this decoder makes about neurons: time, noise, independence, learning rule, binary output... Which ones are wrong for a real brain?

In [ ]:
# Your investigation starts here.

---

## 6. Your bridge

The course used one tool all week: the epistemological bridge. Fill it **for your decoder**: for each piece of biological reality, what did your pipeline turn it into, and was it kept, simplified or discarded?

| Biological reality | What your model turned it into | Kept, simplified or discarded? |
|:---|:---|:---|
| Spike trains over time | ... | ... |
| Synaptic strength and sign | ... | ... |
| Each neuron's identity (channel) | ... | ... |
| Trial-to-trial variability (noise) | ... | ... |
| How the brain learns | ... | ... |
| *(add your own rows)* | ... | ... |

---

## Bonus: what would need to change in the pipeline to account for what was discarded?

Your decoder only sees spike **counts**. Design a feature that uses information the counts throw away, **still fed to the same Perceptron**, and show on the test trials what it gains.

*Hints*: the PSTH cuts time into bins (Chapter 3). Counting spikes in several time windows instead of one gives a longer feature vector. The time of the first spike after the stimulus is also a number.

In [ ]:
# Bonus: your new features, the same Perceptron, a fair comparison on the test trials

<div style="border-left: 4px solid #f59e0b; padding: 0.5em 1em; background: rgba(245,158,11,0.08);">
<b>Deliverables, end of November</b>: a <b>mini-article</b> (4 pages, see <code>REPORT_TEMPLATE.md</code>) with this notebook, cleaned, in annex, and its <b>30-minute presentation</b> at the lab meeting. Both show your decoder's performance <b>on held-out trials</b>, <b>at least two figures</b> supporting your interpretation, <b>your bridge table</b>, and a clear answer to the PI's question: what the model captures about neural coding, and where it stops being a good account of the biology.
</div>